# SHD (Spiking Heidelberg Digits) Tutorial

Training a GLIF RSNN on the Spiking Heidelberg Digits dataset.

**Task:** Classify spoken digits (0-9) from spiking audio input.

**Key features:**
- 700 input channels (cochlear model)
- Pre-spiking input (no encoding needed)
- 20 output classes (digits 0-9 in English and German)

In [ ]:
import torch
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from btorch.models import environ, functional
from btorch.models.init import uniform_v_

from src.model import SingleLayerGLIFRSNN
from src.loss import CombinedLoss, LossConfig
from tutorials.shd import get_dataloaders, get_task_defaults

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load Data

In [ ]:
defaults = get_task_defaults()
print("Task defaults:", defaults)

class Config:
    pass

config = Config()
for k, v in defaults.items():
    setattr(config, k, v)
config.data_dir = '../data'
config.batch_size = 32

train_loader, test_loader, input_dim, output_dim, T = get_dataloaders(config)

print(f"\nData loaded:")
print(f"  Input dim: {input_dim}")
print(f"  Output dim: {output_dim}")
print(f"  Timesteps: {T}")

## 2. Visualize Spiking Input

In [ ]:
x_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: {x_batch.shape}")

x_sample = x_batch[:, 0, :].cpu()

fig, ax = plt.subplots(figsize=(12, 6))

# Show subset of channels
channels = torch.where(x_sample > 0)
ax.scatter(channels[0].numpy(), channels[1].numpy(), c='black', s=1)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Channel')
ax.set_title(f'SHD Input Spikes (Label: {y_batch[0].item()})')
plt.tight_layout()
plt.show()

## 3. Train

In [ ]:
model = SingleLayerGLIFRSNN(
    input_dim=input_dim,
    output_dim=output_dim,
    n_neuron=256,
    n_adapt=128,
    dt=defaults['dt'],
).to(device)

# Init state
functional.init_net_state(model.rnn, batch_size=config.batch_size, device=device)
uniform_v_(model.neuron, set_reset_value=True)

# Setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = CombinedLoss(LossConfig(), dt=defaults['dt'])

# Train
model.train()
for batch_idx, (x, target) in enumerate(train_loader):
    x, target = x.to(device), target.to(device)
    
    functional.reset_net(model.rnn, batch_size=x.shape[1])
    optimizer.zero_grad()
    
    with environ.context(dt=defaults['dt']):
        output, states = model(x)
    
    loss, _ = loss_fn(output, target, states, T)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    if batch_idx % 10 == 0:
        pred = output.argmax(dim=1)
        acc = pred.eq(target).float().mean().item() * 100
        print(f"Batch {batch_idx}: loss={loss.item():.4f}, acc={acc:.2f}%")
    
    if batch_idx >= 50:
        break

## 4. Visualize Hidden Activity

In [ ]:
model.eval()

with torch.no_grad():
    x, target = next(iter(test_loader))
    x = x.to(device)
    
    functional.reset_net(model.rnn, batch_size=x.shape[1])
    
    with environ.context(dt=defaults['dt']):
        output, states = model(x)
    
    spikes = states['spikes'].cpu()[:, 0, :]

fig, ax = plt.subplots(figsize=(12, 6))

for neuron_idx in range(min(100, model.n_neuron)):
    spike_times = torch.where(spikes[:, neuron_idx] > 0)[0]
    ax.scatter(spike_times.numpy(), [neuron_idx] * len(spike_times), 
               c='black', s=5, alpha=0.6)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron Index')
ax.set_title('SHD Recurrent Layer Spikes')
plt.tight_layout()
plt.show()

print(f"Mean firing rate: {spikes.sum() / (T * model.n_neuron) * 1000:.2f} Hz")